In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Read the CSV file
df = pd.read_csv("data/sex-ratio-data-arbuthnot.csv")

# Show the first rows of the data
print(df.head())

# Optionally, display the whole dataframe
#print(df)

In [ ]:
# Plot boys and girls baptisms over time
plt.figure(figsize=(12,6))
plt.plot(df["year"], df["boys"], label="Boys")
plt.plot(df["year"], df["girls"], label="Girls")
plt.xlabel("Year")
plt.ylabel("Number of Baptisms")
plt.title("Number of Boys and Girls Baptized Each Year")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Create a gender ratio column (boys per girl)
df["gender_ratio"] = df["boys"] / df["girls"]

# Compute a moving average (10-year window by default)
df["ratio_ma"] = df["gender_ratio"].rolling(window=10, center=True).mean()

# Plot gender ratio with moving average
plt.figure(figsize=(12,6))
plt.plot(df["year"], df["gender_ratio"], color="lightgray", label="Yearly Ratio")
plt.plot(df["year"], df["ratio_ma"], color="blue", linewidth=2, label="10-Year Moving Average")
plt.axhline(1, color="red", linestyle="--", label="Equal Ratio (1.0)")
plt.xlabel("Year")
plt.ylabel("Gender Ratio (Boys/Girls)")
plt.title("Gender Ratio of Baptisms Over Time")
plt.legend()
plt.grid(True)
plt.show()

# Arbuthnot's Sign Test (1710)

**Observation:**
In every year, more boys than girls were baptized.

**Null Hypothesis (H0):**
Each year, boys and girls are equally likely (p = 0.5).

**Argument:**
- The probability that boys > girls in a single year is 0.5 under H0.
- Since all n years show more boys, the probability of this happening by chance is:
      P = 0.5^n

In [ ]:
n_years = len(df)
k_more_boys = (df["boys"] > df["girls"]).sum()
print(f"Years: {n_years}, Years with more boys: {k_more_boys}")
assert n_years == k_more_boys

# Probability all years have more boys
p = 0.5**n_years

print(f"Probability all years favor boys under H0: {p:.3e}")

In [ ]:
# Recovering Arbuthnot's computation
print(f"{2**82:.3e}")

![](data/arbuthnot.png)


# What if this experiment was done on another planet?

In [ ]:
# Creating new (fake) data
# Copy to a new DataFrame
df_mars = df.copy()

# Number of random years to swap
num_swap = 35  # adjust as needed

# Randomly select indices to swap
swap_indices = np.random.choice(df_mars.index, size=num_swap, replace=False)

# Swap boys and girls counts for the selected years
df_mars.loc[swap_indices, ["boys", "girls"]] = df_mars.loc[swap_indices, ["girls", "boys"]].values

print(f"Swapped boys and girls in {num_swap} random years for df_mars:")
print(df_mars.loc[swap_indices, ["year", "boys", "girls"]])

In [ ]:
# Plot boys and girls baptisms over time
plt.figure(figsize=(12,6))
plt.plot(df["year"], df_mars["boys"], label="Boys")
plt.plot(df["year"], df_mars["girls"], label="Girls")
plt.xlabel("Year")
plt.ylabel("Number of Baptisms on Mars")
plt.title("Number of Boys and Girls Baptized Each Year on Mars")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Create a gender ratio column (boys per girl)
df_mars["gender_ratio"] = df_mars["boys"] / df_mars["girls"]

# Compute a moving average (10-year window by default)
df_mars["ratio_ma"] = df_mars["gender_ratio"].rolling(window=10, center=True).mean()

# Plot gender ratio with moving average
plt.figure(figsize=(12,6))
plt.plot(df_mars["year"], df_mars["gender_ratio"], color="lightgray", label="Yearly Ratio")
plt.plot(df_mars["year"], df_mars["ratio_ma"], color="blue", linewidth=2, label="10-Year Moving Average")
plt.axhline(1, color="red", linestyle="--", label="Equal Ratio (1.0)")
plt.xlabel("Year")
plt.ylabel("Gender Ratio (Boys/Girls) on Mars")
plt.title("Gender Ratio of Baptisms Over Time on Mars")
plt.legend()
plt.grid(True)
plt.show()

## Arbuthnot's Sign Test (General Case)

**Observation:**  
In each year, the number of boys and girls baptized is recorded. Some years have more boys, some years more girls.

**Null Hypothesis (H₀):**  
Each year, boys and girls are equally likely to be more numerous (probability = 0.5).

**Procedure:**  
1. Count the total number of years `n`.  
2. Count the number of years `k` where boys > girls.  
3. Under H₀, the number $X$ of “boy-favored” years follows a Binomial(n, 0.5) distribution.  
4. Compute the probability of observing **k or more boy-favored years** using a one-sided binomial test:

\begin{array}
\mathbb{P}(X \geq k) = \sum_{i=k}^{n} {n\choose i} (0.5)^n
\end{array}

In [ ]:
from scipy.stats import binomtest, binom

# Arbuthnot's sign test
n_years = len(df_mars)
k_success = (df_mars["boys"] > df_mars["girls"]).sum()
p_value = binomtest(k_success, n_years, p=0.5, alternative="greater").pvalue

print(f"Years: {n_years}, Years with more boys (on Mars): {k_success}")
print(f"Binomial test p-value = {p_value:.3e}")

# Total number of years
n = n_years

# Possible number of boy-favored years
k_values = np.arange(0, n + 1)

# Binomial probabilities
probabilities = binom.pmf(k_values, n, 0.5)

# Plot histogram
plt.figure(figsize=(12,6))
bars = plt.bar(k_values, probabilities, color="skyblue", edgecolor="black")

# Shade bars for k >= k_success (one-sided p-value region)
for k in range(k_success, n + 1):
    bars[k].set_color("red")

plt.xlabel("Number of boy-favored years (k)")
plt.ylabel("Probability")
plt.title(f"Binomial distribution (n={n}, p=0.5)\nRed bars: k ≥ {k_success} (p-value region)")
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.show()

**Interpretation:**  
- A small p-value indicates that the observed excess of boys is highly unlikely under random chance.  
- This generalizes Arbuthnot's original argument to datasets where **some years have more girls than boys**.

Done with the help of ChatGPT.